# 1. Import Ntuple and DecayHash

In [1]:
import glob
import numpy as np
import pandas
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm
plt.style.use('belle2')
from tqdm import tqdm
import pyhf
pyhf.set_backend('numpy','minuit')
plt.rcParams["axes.prop_cycle"] = plt.cycler("color", plt.cm.tab20.colors)

In [2]:
def statistics(df):
    counts=df.count()
    mean=df.mean()
    std=df.std()
    return f'''counts = %d \nmean = %5.3f \nstd = %5.3f''' %(counts,mean,std)

kwarg={'histtype':'step','lw':2}
    

plt.rcParams["axes.prop_cycle"] = plt.cycler("color", plt.cm.tab20.colors)

def plot_all_signals(cut, variable):
    fig,axs =plt.subplots(2,3,figsize=(16,10), sharex=True, sharey=False)
    fig.suptitle(f'All signals with {cut}')
    fig.supylabel('# of candidates per bin',x=0.06)
    fig.supxlabel(f'{variable}', y=0.06)
    i=0
    j=0
    for sample_name, sample in samples.items():
        (counts, bins) = np.histogram(sample.query(cut)[variable], bins=50)
        if sample_name in ['sig_D_l_nu','sig_D_tau_nu','bkg_fakeDTC','bkg_fakeB','bkg_continuum','bkg_others']:
            factor = 1
        elif sample_name in ['sig_Dst_l_nu','sig_Dst_tau_nu','all_Dstst_l_nu','all_Dstst_tau_nu']:
            factor = 1
        axs[i,j].hist(bins[:-1], bins, weights=factor*counts,label=sample_name,**kwarg)

        #plt.legend(bbox_to_anchor=(1,1),ncol=3, fancybox=True, shadow=True,labelspacing=1.5)
        axs[i,j].grid()
        axs[i,j].set_title(sample_name)
        j+=1
        if j==3:
            i+=1
            j=0
            
def plot_all_signals_2d(cut):
    variable_x = 'B0_CMS3_weMissM2'
    variable_y = 'p_D_l'
    xedges = np.linspace(-2, 10, 48)
    yedges = np.linspace(0.4, 4.6, 42)

    n_rows,n_cols = [2,3]
    fig,axs=plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(16,8),sharex=True, sharey='all')
    fig.suptitle('Signals')
    fig.supylabel('$|p_D|\ +\ |p_l|\ [GeV]$', x=0.05)
    fig.supxlabel('$M_{miss}^2\ [GeV^2/c^4]$')
    i=0
    j=0
    for name, sample in samples.items():
        (counts, xedges, yedges) = np.histogram2d(sample.query(cut)[variable_x], 
                                              sample.query(cut)[variable_y],
                                              bins=[xedges, yedges])
        counts = counts.T + 0.01
        X, Y = np.meshgrid(xedges, yedges)
        im=axs[i,j].pcolormesh(X, Y, counts, cmap='jet', norm=colors.LogNorm())
        axs[i,j].grid()
        axs[i,j].set_xlim(xedges.min(),xedges.max())
        axs[i,j].set_ylim(yedges.min(),yedges.max())
        axs[i,j].set_title(name,fontsize=12)
        fig.colorbar(im,ax=axs[i,j])
        j+=1
        if j==3:
            i+=1
            j=0
            
def plot_overlaid_signals(cut, variable):
    fig,axs =plt.subplots(1,2,figsize=(12,5), sharex=True, sharey=False)
    fig.suptitle(f'Overlaid signals with pre-selection', y=1)
    fig.supylabel('# of candidates per bin',x=0.06)
    #fig.supxlabel('$|\\vec{p_D}|\ +\ |\\vec{p_l}|$  [GeV/c]')
    #fig.supxlabel('$M_{miss}^2 \ [GeV^2/c^4]$')
    fig.supxlabel(f'{variable}')

    for sample_name, sample in samples.items():
        (counts, bins) = np.histogram(sample.query(cut)[variable], bins=50)
        factor=1
        if sample_name in ['sig_D_tau_nu','sig_Dst_tau_nu','all_Dstst_tau_nu']:
            axs[0].hist(bins[:-1], bins, weights=factor*counts,label=sample_name,**kwarg)
            axs[0].legend()
        elif sample_name in ['sig_D_l_nu','sig_Dst_l_nu','all_Dstst_l_nu']:
            axs[1].hist(bins[:-1], bins, weights=factor*counts,label=sample_name,**kwarg)
            axs[1].legend()
        #plt.legend(bbox_to_anchor=(1,1),ncol=3, fancybox=True, shadow=True,labelspacing=1.5)

    axs[0].set_title('signals')
    axs[1].set_title('normalization')
    axs[0].grid()
    axs[1].grid()
    
def plot_overlaid_signals_2(cut, variable):
    fig,axs =plt.subplots(figsize=(12,5), sharex=True, sharey=False)
    fig.suptitle(f'Overlaid signals with pre-selection', y=1)
    fig.supylabel('# of candidates per bin',x=0.06)
    #fig.supxlabel('$|\\vec{p_D}|\ +\ |\\vec{p_l}|$  [GeV/c]')
    #fig.supxlabel('$M_{miss}^2 \ [GeV^2/c^4]$')
    fig.supxlabel(f'{variable}')

    for sample_name, sample in samples.items():
        if sample_name in ['sig_D_tau_nu']:
            (counts, bins) = np.histogram(sample.query(cut)[variable], bins=50)
            factor=1
        elif sample_name in ['all_Dstst_l_nu']:
            (counts, bins) = np.histogram(sample.query(cut)[variable], bins=50)
            factor=1
        else:
            continue
        axs.hist(bins[:-1], bins, weights=factor*counts,label=sample_name,**kwarg)
        axs.legend()
        #plt.legend(bbox_to_anchor=(1,1),ncol=3, fancybox=True, shadow=True,labelspacing=1.5)
    axs.grid()
    
def plot_projection(cut,variable):
    fig,axs =plt.subplots(sharex=True, sharey=False)
    for sample_name, sample in samples.items():
        (counts, bins) = np.histogram(sample.query(cut)[variable], bins=50)
        factor=1
        if sample_name in ['sig_D_tau_nu','sig_Dst_tau_nu','all_Dstst_tau_nu']:
            axs.hist(bins[:-1], bins, weights=factor*counts,label=f'{sample_name} \n{statistics(sample.query(cut)[variable])}',**kwarg)
        elif sample_name in ['sig_D_l_nu','sig_Dst_l_nu','all_Dstst_l_nu']:
            axs.hist(bins[:-1], bins, weights=factor*counts,label=f'{sample_name} \n{statistics(sample.query(cut)[variable])}',**kwarg)

    axs.set_title('Overlaid signals with pre-selection')
    axs.set_xlabel(f'{variable}')
    axs.set_ylabel('# of candidates per bin')
    axs.grid()
    plt.legend(bbox_to_anchor=(1,1),ncol=3, fancybox=True, shadow=True,labelspacing=1.5)
    
    
def plot_fitting_difference(yaml_file):
    fig,axs =plt.subplots(2,3,figsize=(16,10), sharex=True, sharey=False)
    fig.suptitle(f'fitted yield - true yield')
    fig.supylabel('yield difference',x=0.06)
    fig.supxlabel(f'index of subset samples', y=0.06)
    i=0
    j=0
    with open(yaml_file, 'r+') as f:
        data = yaml.safe_load(f)
        components = data['signal_e']

    for comp_name, info in components.items():
        axs[i,j].errorbar(x=range(1,len(info['difference'])+1), y=info['difference'], yerr=info['errors'], fmt='ko')
        axs[i,j].axhline(y=0, linestyle='-', linewidth=3, color='r')
        axs[i,j].grid()
        axs[i,j].set_title(comp_name)
        j+=1
        if j==3 and i==0:
            i+=1
            j=0
        if j==3 and i==1:
            break

In [9]:
# read in root-file as a pandas dataframe
Dstst_e_nu_selection = 'DecayMode=="all_Dstst_e_nu" and D_mcPDG*e_mcPDG==411*11 and e_genMotherPDG==B0_mcPDG and \
    ((B0_mcErrors<64 and B0_mcPDG*D_mcPDG==-511*411) or (B0_mcErrors<512 and abs(B0_mcPDG)==521))'
Dstst_tau_nu_selection = 'DecayMode=="all_Dstst_tau_nu" and D_mcPDG*e_mcPDG==411*11 and e_mcPDG*e_genMotherPDG==11*15 and \
    ((B0_mcErrors<64 and B0_mcPDG*D_mcPDG==-511*411) or (B0_mcErrors<512 and abs(B0_mcPDG)==521))'
signals_selection = 'B0_mcPDG*D_mcPDG==-511*411 and D_mcPDG*e_mcPDG==411*11 and e_mcPDG*e_genMotherPDG==11*15'
norms_selection = 'B0_mcPDG*D_mcPDG==-511*411 and D_mcPDG*e_mcPDG==411*11 and e_genMotherPDG==B0_mcPDG'

folder = '/home/belle/zhangboy/R_D/Generic_MC14ri/test_sigDDst_e_bengal_1'
pfs = glob.glob(f"{folder}/test_sigDDst_e_bengal_1_0.parquet")

samples = {}

df_bestSelected = pandas.read_parquet(pfs, engine="pyarrow")
#df2 = pandas.read_feather("../Ntuples/bengal_e_10k_Test2.feather")
#df_charged = root_pandas.read_root(charged,key='B0')
#data = pandas.concat([df_mixed,df_charged], ignore_index = True)

# Signal components
sig_D_tau_nu=df_bestSelected.query(f'DecayMode=="sig_D_tau_nu" and B0_mcErrors<32 and {signals_selection}').copy()
sig_Dst_tau_nu=df_bestSelected.query(f'DecayMode=="sig_Dst_tau_nu" and B0_mcErrors<64 and {signals_selection}').copy()
sig_D_e_nu=df_bestSelected.query(f'DecayMode=="sig_D_e_nu" and B0_mcErrors<16 and {norms_selection}').copy()
sig_Dst_e_nu=df_bestSelected.query(f'DecayMode=="sig_Dst_e_nu" and B0_mcErrors<64 and {norms_selection}').copy() 
all_Dstst_tau_nu=df_bestSelected.query(Dstst_tau_nu_selection).copy() 
all_Dstst_e_nu=df_bestSelected.query(Dstst_e_nu_selection).copy()

cut = 'e_p>0'
samples['sig_D_tau_nu'] = sig_D_tau_nu.query(cut)
samples['sig_Dst_tau_nu'] = sig_Dst_tau_nu.query(cut)
samples['sig_D_l_nu'] = sig_D_e_nu.query(cut)
samples['sig_Dst_l_nu'] = sig_Dst_e_nu.query(cut)
samples['all_Dstst_tau_nu'] = all_Dstst_tau_nu.query(cut)
samples['all_Dstst_l_nu'] = all_Dstst_e_nu.query(cut)

In [10]:
for name in samples:
    print(f'{name}: {len(samples[name])}')

sig_D_tau_nu: 119066
sig_Dst_tau_nu: 73953
sig_D_l_nu: 54021
sig_Dst_l_nu: 40994
all_Dstst_tau_nu: 538
all_Dstst_l_nu: 18402


In [8]:
for name in samples:
    print(f'{name}: {len(samples[name])}')

sig_D_tau_nu: 96237
sig_Dst_tau_nu: 61287
sig_D_l_nu: 52388
sig_Dst_l_nu: 40312
all_Dstst_tau_nu: 438
all_Dstst_l_nu: 17105


In [41]:
df2 = df_bestSelected.query('DecayMode=="all_Dstst_e_nu" and D_mcPDG*e_mcPDG==411*11 and e_genMotherPDG==B0_mcPDG and \
    ((B0_mcErrors<64 and B0_mcPDG*D_mcPDG==-511*411) or (B0_mcErrors<512 and abs(B0_mcPDG)==521))')
df3 = df_bestSelected.query('DecayMode=="all_Dstst_e_nu"')
df4 = pandas.concat([df2,df3]).drop_duplicates(keep=False)
len(df4)

5099

In [23]:
df_bestSelected.DecayMode.value_counts()

bkg                 579118
sig_D_e_nu           48330
sig_Dst_e_nu         36679
all_Dstst_e_nu       32161
sig_D_tau_nu          1928
sig_Dst_tau_nu        1205
all_Dstst_tau_nu      1137
all_Dstst_mu_nu        140
sig_D_mu_nu             14
sig_Dst_mu_nu            9
Name: DecayMode, dtype: int64

In [14]:
data["B0_DecayHashEx"].isnull().sum()

5290

In [ ]:
import pandas
pandas.set_option('display.max_rows', None)
print(data.isna().sum())

In [ ]:
cut='DecayMode=="all_Dstst_e_nu" and D_mcPDG*e_mcPDG!=411*11 and e_genMotherPDG==B0_mcPDG and \
    ((B0_mcErrors<64 and B0_mcPDG*D_mcPDG==-511*411) or (B0_mcErrors<512 and abs(B0_mcPDG)==521))'#'DecayMode=="bkg" and B0_isContinuumEvent!=1'
candidate12 = df_cut.query(cut).iloc[2][['B0_DecayHash', "B0_DecayHashEx"]].values

# print the original decay as simulated in MC with removed Bremsstrahlung gammas
print("Monte Carlo Decay with removed Bremsstrahlung gammas: ")
org2 = hashmap2.get_original_decay(*candidate12)
print(org2.to_string())

In [ ]:
samples = {}
names = ['BC','AC']
cut = 'D_vtxReChi2<13 and B0_vtxReChi2<14 and -3.2<B0_deltaE<0 and e_CMS_p>0.2 and \
    5<B0_roeMbc_my_mask and 4.3<B0_CMS2_weMbc and \
    -5<B0_roeDeltae_my_mask<2 and -3<B0_CMS0_weDeltae<2 and \
    abs(B0_roeCharge_my_mask)<3 and nElectrons90+nMuons90==1'
for name in names:
    if name == 'BC':
        df = data.copy()
    else:
        df = data.query(cut).copy()
    
    print(f'{name} before BCS')
    print(df.DecayMode.value_counts())

    df_bestSelected=df.loc[df.groupby(['__experiment__','__run__','__event__','__production__']).B_D_ReChi2.idxmin()]

    print(f'{name} after BCS')
    print(df_bestSelected.DecayMode.value_counts())
    
    df_merged = pandas.merge(df_bestSelected,MC,on=['__event__'],validate='1:1')
    samples[name] = df_merged

# 4. Get fitting templates

In [ ]:
# apply BDTs
# plot mm2, mm2 vs p_D_l
# save the templates

In [ ]:
import basf2_mva
import pandas

identifier_1 = '/home/belle/zhangboy/R_D/Generic_MC14rd/Continuum_Suppression/MVA1_FastBDT.xml'
test_1 = '../Ntuples/bengal_eidglobal_50k_cut.root'
data.query(cut).to_root(test_1, key='B0')
output_file_1 = '../Ntuples/bengal_eidglobal_50k_MVA1.root'

identifier_1_5 = '/home/belle/zhangboy/R_D/Generic_MC14rd/B_bkg_Suppression/MVA1_5/MVA1_5_FastBDT.xml'
test_1_5 = '../Ntuples/bengal_eidglobal_50k_cut.root'
output_file_1_5 = '../Ntuples/bengal_eidglobal_50k_MVA1_5.root'

output_file_1_5_applied = '../Ntuples/bengal_eidglobal_50k_MVA1_5_applied.root'

identifier_2_1 = '/home/belle/zhangboy/R_D/Generic_MC14rd/B_bkg_Suppression/MVA2/MVA2_1_FastBDT.xml'
test_2_1 = output_file_1_5_applied
output_file_2_1 = '../Ntuples/bengal_eidglobal_50k_MVA2_1.root'
output_file_2_1_applied = '../Ntuples/bengal_eidglobal_50k_MVA2_1_applied.root'

In [ ]:
# apply CS BDT identifier_1, merge data file and mva output, rename the column
basf2_mva.expert(basf2_mva.vector(identifier_1),  # weightfile
                 basf2_mva.vector(test_1),
                 'B0', output_file_1)

df1 = data.query(cut).drop_duplicates(subset=['__experiment__','__run__','__event__','__production__','__candidate__']).reset_index(drop=True)
df2 = root_pandas.read_root(output_file_1)
print(len(df1)==len(df2))
df_1 = pandas.concat([df1,df2],axis=1)

df_1=df_1.rename(columns={"__slhome__slbelle__slzhangboy__slR_D__slGeneric_MC14rd__slContinuum_Suppression__slMVA1_FastBDT__ptxml": "MVA1_output"})
df_1=df_1.drop(columns=['__slhome__slbelle__slzhangboy__slR_D__slGeneric_MC14rd__slContinuum_Suppression__slMVA1_FastBDT__ptxml_isSignal'])

# apply BDT 1_5 identifier_1_5, merge, rename, change the output type, save
basf2_mva.expert(basf2_mva.vector(identifier_1_5),  # weightfile
                 basf2_mva.vector(test_1_5),
                 'B0', output_file_1_5)

df3 = root_pandas.read_root(output_file_1_5)
print(len(df_1)==len(df3))
df_1_5 = pandas.concat([df_1,df3],axis=1)

df_1_5=df_1_5.rename(columns={"__slhome__slbelle__slzhangboy__slR_D__slGeneric_MC14rd__slB_bkg_Suppression__slMVA1_5__slMVA1_5_FastBDT__ptxml": "MVA1_5_output"})
df_1_5=df_1_5.drop(columns=['__slhome__slbelle__slzhangboy__slR_D__slGeneric_MC14rd__slB_bkg_Suppression__slMVA1_5__slMVA1_5_FastBDT__ptxml_isSignal'])

df_1_5.MVA1_5_output=np.float64(df_1_5.MVA1_5_output)
print(type(df_1_5.MVA1_5_output[0]))
print(type(df_1_5.isSignal[0]))

df_1_5.to_root(output_file_1_5_applied, key='B0')

# apply BDT 2_1 identifier_2_1, merge, rename, save
basf2_mva.expert(basf2_mva.vector(identifier_2_1),  # weightfile
                 basf2_mva.vector(test_2_1),
                 'B0', output_file_2_1)

df4 = root_pandas.read_root(output_file_2_1)
print(len(df_1_5)==len(df4))
df_2_1 = pandas.concat([df_1_5, df4],axis=1)
print(len(df_1_5)==len(df_2_1))

df_2_1=df_2_1.rename(columns={"__slhome__slbelle__slzhangboy__slR_D__slGeneric_MC14rd__slB_bkg_Suppression__slMVA2__slMVA2_1_FastBDT__ptxml": "MVA2_1_output"})
df_2_1=df_2_1.drop(columns=['__slhome__slbelle__slzhangboy__slR_D__slGeneric_MC14rd__slB_bkg_Suppression__slMVA2__slMVA2_1_FastBDT__ptxml_isSignal'])

df_2_1.to_root(output_file_2_1_applied, key='B0')

In [ ]:
df_2_1.columns
len(df_2_1)

In [ ]:
df_bestSelected=df_2_1.loc[df_2_1.groupby(['__experiment__','__run__','__event__','__production__']).B_D_ReChi2.idxmin()]

In [ ]:
import pandas
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm as cm
plt.style.use('belle2')
# plot mm2, mm2 vs p_D_l
# save the templates

In [ ]:
# Signal components
sig_D_e_nu=samples['AC'].query('DecayMode=="sig_D_e_nu" and B0_mcErrors<16 and abs(D_mcPDG)==411 and abs(e_genMotherPDG)==511').copy()
sig_D_tau_nu=samples['AC'].query('DecayMode=="sig_D_tau_nu" and B0_mcErrors<32 and abs(D_mcPDG)==411 and abs(e_genMotherPDG)==15').copy()
sig_Dst_e_nu=samples['AC'].query('DecayMode=="sig_Dst_e_nu" and B0_mcErrors<64 and abs(D_mcPDG)==411 and abs(e_genMotherPDG)==511').copy()
sig_Dst_tau_nu=samples['AC'].query('DecayMode=="sig_Dst_tau_nu" and B0_mcErrors<64 and abs(D_mcPDG)==411 and abs(e_genMotherPDG)==15').copy()
all_Dstst_e_nu=samples['AC'].query('DecayMode=="all_Dstst_e_nu" and abs(D_mcPDG)==411 and \
((B0_mcErrors<64 and abs(e_genMotherPDG)==511) or (B0_mcErrors<512 and abs(e_genMotherPDG)==521))').copy()
all_Dstst_tau_nu=samples['AC'].query('DecayMode=="all_Dstst_tau_nu" and abs(D_mcPDG)==411 and abs(e_genMotherPDG)==15 and \
((B0_mcErrors<64 and abs(B0_mcPDG)==511) or (B0_mcErrors<512 and abs(B0_mcPDG)==521))').copy()

#sig_D_mu_nu=samples['AC'].query('DecayMode=="sig_D_mu_nu" and B0_mcErrors<16').copy()
#sig_Dst_mu_nu=samples['AC'].query('DecayMode=="sig_Dst_mu_nu" and (16<=B0_mcErrors<32 or B0_mcErrors<8)').copy()
#all_Dstst_mu_nu=samples['AC'].query('DecayMode=="all_Dstst_mu_nu" and (16<=B0_mcErrors<64 or B0_mcErrors<8)').copy()

# Bkg components
bkg_fakeD = samples['AC'].query('abs(D_mcPDG)!=411 and B0_mcErrors!=512 and B0_isContinuumEvent!=1').copy()
bkg_fakeTracksClusters = samples['AC'].query('B0_mcErrors==512 and B0_isContinuumEvent!=1').copy()
bkg_fakeDTC = pandas.concat([bkg_fakeD, bkg_fakeTracksClusters])

bkg_combinatorial = samples['AC'].query('B0_mcPDG==300553 and abs(D_mcPDG)==411 and B0_mcErrors!=512 and B0_isContinuumEvent!=1').copy()
bkg_sigOtherBDTaudecay = samples['AC'].query('(DecayMode=="bkg" or DecayMode=="sig_D_mu_nu" or DecayMode=="sig_Dst_mu_nu" or DecayMode=="all_Dstst_mu_nu") and \
B0_mcPDG!=300553 and abs(D_mcPDG)==411 and B0_mcErrors!=512 and B0_isContinuumEvent!=1').copy()
bkg_fakeB = pandas.concat([bkg_combinatorial, bkg_sigOtherBDTaudecay])

bkg_continuum = samples['AC'].query('B0_isContinuumEvent==1').copy()

bkg_others = pandas.concat([samples['AC'],
                           sig_D_e_nu,
                           sig_D_tau_nu,
                           sig_Dst_e_nu,
                           sig_Dst_tau_nu,
                           all_Dstst_e_nu,
                           all_Dstst_tau_nu,
                           bkg_fakeDTC,
                           bkg_fakeB,
                           bkg_continuum]).drop_duplicates(keep=False)
# Weird! the bkg_others contains some events with
# correct sig decay hash chain and correct B0_mcPDG, D_mcPDG, e_genMotherPDG,
# but with 128< B0_mcErrors < 256 (misID)


def statistics(df):
    counts=df.count()
    mean=df.mean()
    std=df.std()
    return f'''counts = %d \nmean = %5.3f \nstd = %5.3f''' %(counts,mean,std)

kwarg={'bins':50, 'histtype':'step','lw':2}

    
def plot_projection(cut,variable):
    (counts1, bins1) = np.histogram(sig_D_tau_nu.query(cut)[variable], bins=50)
    (counts2, bins2) = np.histogram(sig_Dst_tau_nu.query(cut)[variable], bins=50)
    factor = 1
    plt.hist(bins1[:-1], bins1, weights=factor*counts1,label=f'D_tau_nu \n{statistics(sig_D_tau_nu.query(cut)[variable])}',alpha=0.6)
    sig_D_e_nu.query(cut)[variable].hist(label=f'D_e_nu \n{statistics(sig_D_e_nu.query(cut)[variable])}',**kwarg)
    
    plt.hist(bins2[:-1], bins2, weights=factor*counts2,label=f'Dst_tau_nu \n{statistics(sig_Dst_tau_nu.query(cut)[variable])}',alpha=0.6,histtype='step',lw=2)
    sig_Dst_e_nu.query(cut)[variable].hist(label=f'Dst_e_nu \n{statistics(sig_Dst_e_nu.query(cut)[variable])}',**kwarg)
    all_Dstst_tau_nu.query(cut)[variable].hist(label=f'all_Dstst_tau_nu \n{statistics(all_Dstst_tau_nu.query(cut)[variable])}',**kwarg)
    all_Dstst_e_nu.query(cut)[variable].hist(label=f'all_Dstst_e_nu \n{statistics(all_Dstst_e_nu.query(cut)[variable])}',**kwarg)
    bkg_fakeDTC.query(cut)[variable].hist(label=f'bkg_fakeD_Tracks_Clusters \n{statistics(bkg_fakeDTC.query(cut)[variable])}',**kwarg)
    bkg_fakeB.query(cut)[variable].hist(label=f'bkg_comb_wrongDecay \n{statistics(bkg_fakeB.query(cut)[variable])}',**kwarg)
    bkg_continuum.query(cut)[variable].hist(label=f'bkg_continuum \n{statistics(bkg_continuum.query(cut)[variable])}',**kwarg)
    bkg_others.query(cut)[variable].hist(label=f'bkg_others \n{statistics(bkg_others.query(cut)[variable])}',**kwarg)
    plt.legend(bbox_to_anchor=(1,1),ncol=3, fancybox=True, shadow=True,labelspacing=1.5)

def plot_q2_efficiency(cut, q2='B0_CMS3_weQ2lnuSimple'):
    sig_D_e_nu_BC=samples['BC'].query('DecayMode=="sig_D_e_nu" and B0_mcErrors<16 and abs(D_mcPDG)==411 and abs(e_genMotherPDG)==511').copy()
    sig_D_tau_nu_BC=samples['BC'].query('DecayMode=="sig_D_tau_nu" and B0_mcErrors<32 and abs(D_mcPDG)==411 and abs(e_genMotherPDG)==15').copy()
    (counts1, bins1) = np.histogram(sig_D_e_nu_BC.query(cut)[q2], bins=15)
    (counts2, bins2) = np.histogram(sig_D_e_nu.query(cut)[q2], bins=bins1)
    
    efficiency = counts2 / counts1
    efficiency_err = efficiency * np.sqrt(1/counts1 + 1/counts2)
    #factor = 1
    #plt.hist(bins1[:-1], bins1, weights=factor*efficiency,label='D_l_nu efficiency in q2',histtype='step')
    bin_centers = (bins1[:-1] + bins1[1:]) /2
    plt.errorbar(x=bin_centers, y=efficiency, yerr=efficiency_err, fmt='ko',label='D_l_nu efficiency in q2')
    plt.legend()
    
plt.rcParams["axes.prop_cycle"] = plt.cycler("color", plt.cm.tab20.colors)

In [ ]:
plot_projection('MVA1_output>0.4 and MVA2_1_output>0.2 and B0_roeMbc_my_mask>5.26', 'B0_CMS2_weMissM2')
plt.xlabel("CMS2_weMissM2 $[GeV^2/c^4]$")
plt.ylabel('# of counts per bin')
plt.title('CMS2_weMissM2');
#plt.xlim(-10,10)

In [ ]:
sig_sample = all_Dstst_tau_nu

cut='B0_roeMbc_my_mask>5.26 and MVA1_output>0.4 and MVA2_1_output>0.2'
xedges = np.linspace(0.5, 4, 15)
yedges = np.linspace(-4, 10, 25)
variable_x = 'p_D_l'
variable_y = 'B0_CMS2_weMissM2'

(counts, xedges, yedges) = np.histogram2d(sig_sample.query(cut)[variable_x], 
                                          sig_sample.query(cut)[variable_y],
                                          bins=[xedges, yedges])
counts = counts.T + 0.01
fig,axs=plt.subplots(ncols=1,figsize=(10,4))
X, Y = np.meshgrid(xedges, yedges)
im=axs.pcolormesh(X, Y, counts, cmap='jet', norm=colors.LogNorm())
axs.grid()
fig.colorbar(im)

In [ ]:
import json

workspace_file = '/home/belle/zhangboy/R_D/Signal_MC_ROEx1/2d_2channels_workspace.json'
with open(workspace_file, 'r+') as f:
    data = json.load(f)
    data['channels'][0]['samples'][4]['name'] = 'Dstst_tau_nu'
    data['channels'][0]['samples'][4]['data'] = counts.ravel().tolist()
    # counts.ravel()/.reshape(-1) returns a view, counts.flatten() returns a copy (slower)
    f.seek(0)        # <--- should reset file position to the beginning.
    json.dump(data, f, indent=4)
    f.truncate()     # remove remaining part

In [ ]:
plot_q2_efficiency(cut='-2<B0_CMS3_weQ2lnuSimple<13')
plt.xlabel("q2 $[GeV^2/c^4]$")
plt.ylabel('Cuts Efficiency')
plt.title('Dl$\\nu$ cut efficiency vs. q2');
plt.xlim(-2,12);
plt.ylim(0,1);

In [ ]:
plot_projection('B0_roeMbc_my_mask>5', 'B0_CMS2_weMissM2')
plt.xlabel("CMS2_weMissM2 $[GeV^2/c^4]$")
plt.ylabel('# of counts per bin')
plt.title('MissM2 with E*_roe=$\sqrt{s}/2$')
plt.xlim(-8,8);

In [ ]:
plot_projection('B0_roeMbc_my_mask>5', 'B0_CMS3_weMissM2')
plt.xlabel("CMS3_weMissM2 $[GeV^2/c^4]$")
plt.ylabel('# of counts per bin')
plt.title('MissM2 with E*_roe=$\sqrt{s}/2$ and p*_roe=0')
plt.xlim(-8,8);

In [ ]:
plot_projection('B0_roeMbc_my_mask>5', 'B0_CMS4_weMissM2')
plt.xlabel("CMS4_weMissM2 $[GeV^2/c^4]$")
plt.ylabel('# of counts per bin')
plt.title('MissM2 with E*_roe=$\sqrt{s}/2$ and p*_roe=0.34GeV + direction');
plt.xlim(-8,8);

In [ ]:
plot_projection('B0_roeMbc_my_mask>5', 'q2_MC')
plt.xlabel("q2 $[GeV^2/c^4]$")
plt.ylabel('# of counts per bin')
plt.title('MC True q2')
plt.xlim(-2,13);

In [ ]:
plot_projection('B0_roeMbc_my_mask>5', 'B0_CMS2_weQ2lnuSimple')
plt.xlabel("CMS2_q2 $[GeV^2/c^4]$")
plt.ylabel('# of counts per bin')
plt.title('q2 with E*_roe=$\sqrt{s}/2$')
plt.xlim(-8,13);

In [ ]:
plot_projection('B0_roeMbc_my_mask>5', 'B0_CMS3_weQ2lnuSimple')
plt.xlabel("CMS2_q2 $[GeV^2/c^4]$")
plt.ylabel('# of counts per bin')
plt.title('q2 with E*_roe=$\sqrt{s}/2$ and p*_roe=0')
plt.xlim(-2,13);

In [ ]:
plot_projection('B0_roeMbc_my_mask>5', 'B0_CMS4_weQ2lnuSimple')
plt.xlabel("CMS4_q2 $[GeV^2/c^4]$")
plt.ylabel('# of counts per bin')
plt.title('q2 with E*_roe=$\sqrt{s}/2$ and p*_roe=0.34GeV + direction');
plt.xlim(-4,13);

In [ ]:
plt.scatter(sig_D_e_nu.q2_MC, sig_D_e_nu.B0_CMS2_weQ2lnuSimple,label='q2')
x = np.linspace(-0.5, 12, 10)
plt.plot(x,x,color='red',label='y=x')
plt.xlabel("MC True q2 $[GeV^2/c^4]$")
plt.ylabel('Reconstructed q2')
plt.title('q2 with E*_roe=$\sqrt{s}/2$')
plt.legend();

In [ ]:
plt.scatter(sig_D_e_nu.q2_MC, sig_D_e_nu.B0_CMS3_weQ2lnuSimple,label='q2')
x = np.linspace(-0.5, 12, 10)
plt.plot(x,x,color='red',label='y=x')
plt.xlabel("MC True q2 $[GeV^2/c^4]$")
plt.ylabel('Reconstructed q2')
plt.title('q2 with E*_roe=$\sqrt{s}/2$ and p*_roe=0')
plt.legend();

In [ ]:
plt.scatter(sig_D_e_nu.q2_MC, sig_D_e_nu.B0_CMS4_weQ2lnuSimple,label='q2')
x = np.linspace(-0.5, 12, 10)
plt.plot(x,x,color='red',label='y=x')
plt.xlabel("MC True q2 $[GeV^2/c^4]$")
plt.ylabel('Reconstructed q2')
plt.title('q2 with E*_roe=$\sqrt{s}/2$ and p*_roe=0.34GeV + direction')
plt.legend();

In [ ]:
plot_projection('q2_MC<2', 'B0_CMS3_weQ2lnuSimple')
plt.xlabel("CMS2_q2 $[GeV^2/c^4]$")
plt.ylabel('# of counts per bin')
plt.title('q2 with E*_roe=$\sqrt{s}/2$ and p*_roe=0')
plt.xlim(-2,13);

In [ ]:
plot_projection('2<q2_MC<4', 'B0_CMS3_weQ2lnuSimple')
plt.xlabel("CMS2_q2 $[GeV^2/c^4]$")
plt.ylabel('# of counts per bin')
plt.title('q2 with E*_roe=$\sqrt{s}/2$ and p*_roe=0')
plt.xlim(-2,13);

In [ ]:
plot_projection('4<q2_MC<6', 'B0_CMS3_weQ2lnuSimple')
plt.xlabel("CMS2_q2 $[GeV^2/c^4]$")
plt.ylabel('# of counts per bin')
plt.title('q2 with E*_roe=$\sqrt{s}/2$ and p*_roe=0')
plt.xlim(-2,13);

In [ ]:
plot_projection('6<q2_MC<8', 'B0_CMS3_weQ2lnuSimple')
plt.xlabel("CMS2_q2 $[GeV^2/c^4]$")
plt.ylabel('# of counts per bin')
plt.title('q2 with E*_roe=$\sqrt{s}/2$ and p*_roe=0')
plt.xlim(-2,13);

In [ ]:
plot_projection('8<q2_MC<10', 'B0_CMS3_weQ2lnuSimple')
plt.xlabel("CMS2_q2 $[GeV^2/c^4]$")
plt.ylabel('# of counts per bin')
plt.title('q2 with E*_roe=$\sqrt{s}/2$ and p*_roe=0')
plt.xlim(-2,13);

In [ ]:
plot_projection('10<q2_MC', 'B0_CMS3_weQ2lnuSimple')
plt.xlabel("CMS2_q2 $[GeV^2/c^4]$")
plt.ylabel('# of counts per bin')
plt.title('q2 with E*_roe=$\sqrt{s}/2$ and p*_roe=0')
plt.xlim(-2,13);